In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/pre_cell_26.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 26 ###

train_text_df = train_text_df[:5]

# 1. Tokenize each essay into one row per token, with token indices
tokenized = (
    train_text_df[['id', 'text']]
    .assign(tokens=train_text_df['text'].str.split(' '))
    .explode('tokens')
    .reset_index(drop=True)
)
tokenized['token_index'] = tokenized.groupby('id').cumcount()
print(1)

# 2. Expand the discourse annotations into one row per token index with B-/I- labels
labels = (
    train[['id', 'discourse_type', 'predictionstring']]
    .reset_index()
    .rename(columns={'index': 'disc_row'})
    .assign(
        token_index=lambda df: df['predictionstring'].str.split(' ').apply(lambda lst: list(map(int, lst)))
    )
    .explode('token_index')
)
print(2)
labels['pos_in_discourse'] = labels.groupby('disc_row').cumcount()
labels['label'] = (
    ('B-' + labels['discourse_type'])
    .where(labels['pos_in_discourse'] == 0,
           'I-' + labels['discourse_type'])
)
labels = labels[['id', 'token_index', 'label']]
print(3)

# 3. Merge tokenized text with labels, defaulting missing labels to "O"
tokenized = (
    tokenized
    .merge(labels, on=['id', 'token_index'], how='left')
)
tokenized['label'] = tokenized['label'].fillna('O')
print(4)

# 4. Aggregate back to one list of labels per essay
entities_df = (
    tokenized
    .groupby('id')['label']
    .agg(list)
    .reset_index()
    .rename(columns={'label': 'entities'})
)
print(5)

# 5. Attach the entities column to the original DataFrame
train_text_df = train_text_df.merge(entities_df, on='id', how='left')
print(6)

In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_26.pickle